# NB12 — Training-data weights for stage 2 (SPEC T4)

1. **Confusion table** from Extra folds A + B (what real readers said instead of each expected phoneme).
2. **TTS weights**: each Iqra_TTS clip weighted by how realistic its injected errors are, clipped to [0.5, 3].
3. **Agreement filter**: the stage-1 model decodes all of Iqra_train (train) and Iqra_TTS; a clip whose decode
   differs from its label by PER > 0.30 gets weight 0.2. Bulk decode: batched + fp16, used only for this
   coarse cut-off, never for reported numbers.

Writes `sample_weights.parquet` and `confusion_table.json` for NB13. Never touches QuranMB.v2 or IqraEval_Test_GT.

**Run settings:** Accelerator **GPU T4**, Internet *On*. Add data: `iqra-cache`, the NB11 output,
`quran-phoneme-model` (checkpoint-5000), the NB8 model output (notebook17aed3d165, parity check).
Secrets: `HF_TOKEN`. Set `SMOKE = True` in the next cell for a 10-minute trial on 500 clips per dataset.

**After the run:** Save Version. Paste back everything printed by the last cell (**PASTE BACK**).


In [ ]:
# ---- Shared install cell (identical in nb11-nb16) --------------------------------
# pip install git+... does not fetch submodules, and the official scorer is a pinned
# submodule, so clone recursively and install from the clone (SPEC decision 21).
import os, subprocess, sys
REPO_URL = "https://github.com/markeet04/Quran_phenome.git"
REPO_DIR = "/tmp/Quran_phenome"          # outside /kaggle/working: not saved as output
BRANCH = "main"

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")   # only if the repo is private
except Exception:
    pass
url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL

def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode:
        msg = r.stderr if not token else r.stderr.replace(token, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg}")
    return r.stdout.strip()

if not os.path.isdir(REPO_DIR):
    _git("clone", "--recursive", "-q", "-b", BRANCH, url, REPO_DIR)
else:   # a re-run in the same session: move the existing clone to the latest commit
    _git("-C", REPO_DIR, "fetch", "-q", url, BRANCH)
    _git("-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
_git("-C", REPO_DIR, "submodule", "update", "--init", "--recursive", "-q")
REPO_COMMIT = _git("-C", REPO_DIR, "rev-parse", "HEAD")

# No datasets pin: Kaggle now runs Python 3.13, where datasets 2.21's dependencies break.
# Audio is decoded by iqra.data (soundfile on the raw bytes), so any datasets version works.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
                   capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("pip install failed:\n" + r.stdout[-1500:] + "\n" + r.stderr[-1500:])
sys.path.insert(0, f"{REPO_DIR}/src")
import warnings
warnings.filterwarnings("ignore")
import iqra.align
import datasets, numpy, pandas
print("python", sys.version.split()[0], "| datasets", datasets.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("repo commit:", REPO_COMMIT)
print("official scorer:", iqra.align.mdd_eval_dir())


In [ ]:
# ---- Environment, parity, inputs ----------------------------------------------------------
SMOKE = False                     # True: 500 clips per dataset, to check timing before the full run
BATCH_SECONDS = 160               # audio seconds per padded batch in the bulk decode
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import json, time
import numpy as np
import pandas as pd
import torch, transformers
from transformers import AutoFeatureExtractor
from datasets import load_dataset, load_from_disk
from iqra import data, parity, posteriors, weights
from iqra.config import Config, find_cache_root, find_input
from iqra.decode import greedy_decode
from iqra.vocab import Vocab

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
except Exception as e:
    raise RuntimeError("HF_TOKEN secret is required for the gated Iqra_Extra_IS26") from e

par = parity.verify_environment(DEVICE)          # decision 27; raises on failure
print("version parity: PASS | here", par["here"], "| reference", par["reference"])
cfg = Config()
cfg.paths.cache_root = find_cache_root()
SPLITS_DIR = find_input("splits.json", what="the NB11 output")
CKPT = find_input("checkpoint-5000/config.json", what="quran-phoneme-model (holds checkpoint-5000)")
OUT = "/kaggle/working/nb12_weights"
os.makedirs(OUT, exist_ok=True)
vocab = Vocab.load(cfg.paths.vocab)
ID2TOK, BLANK = vocab.id2tok, vocab.blank_id
splits = data.load_splits(f"{SPLITS_DIR}/splits.json")
report = {"repo_commit": REPO_COMMIT, "smoke": SMOKE, "parity": {"pass": par["pass"], "here": par["here"]},
          "device": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"}


In [ ]:
# ---- 1. Confusion table from Extra folds A + B --------------------------------------------------
extra = load_dataset(cfg.datasets.extra, split="train", token=HF_TOKEN)
ex_ids = [str(x) for x in list(extra["id"])]
assert ex_ids == splits["extra"]["ids_by_row"]
ex_ref = dict(zip(ex_ids, [str(x) for x in list(extra["phoneme_ref"])]))
ex_mis = dict(zip(ex_ids, [str(x) for x in list(extra["phoneme_mis"])]))
fold_ids = splits["extra"]["fold_a"] + splits["extra"]["fold_b"]
assert not set(fold_ids) & set(splits["extra"]["extra_dev"]), "Extra-dev must not feed the confusion table"
table = weights.ConfusionTable(vocab.phonemes())
for cid in fold_ids:
    table.add(ex_ref[cid].split(), ex_mis[cid].split())
with open(f"{OUT}/confusion_table.json", "w", encoding="utf-8") as f:
    json.dump(table.to_json(), f, ensure_ascii=False, indent=1)
top10 = table.top_pairs(10)
n_err = sum(sum(c.values()) for c in table.sub.values())
print(f"confusion table from {len(fold_ids)} fold clips: {n_err} substitution/deletion events, "
      f"{sum(table.ins.values())} insertions")
print("top-10 expected -> said:", [f"{e}->{o} ({n})" for (e, o), n in top10])
report["confusion"] = {"clips": len(fold_ids), "sub_del_events": n_err, "insertions": sum(table.ins.values()),
                       "top10": [[e, o, n] for (e, o), n in top10]}


In [ ]:
# ---- 2. TTS weights (realism of injected errors) -----------------------------------------------
tts = load_from_disk(cfg.paths.tts_arrow)
tts = tts["train"] if hasattr(tts, "keys") and "train" in tts.keys() else tts
tts_ref = [str(x) for x in list(tts["phoneme_ref"])]
tts_mis = [str(x) for x in list(tts["phoneme_mis"])]
n_tts = 500 if SMOKE else len(tts)
tts_w = np.array([weights.tts_weight(tts_ref[i].split(), tts_mis[i].split(), table) for i in range(n_tts)])
q = np.quantile(tts_w, [0.05, 0.25, 0.5, 0.75, 0.95])
print(f"TTS weights over {n_tts} clips: error-free {(tts_w == 1.0).mean():.1%} | at floor 0.5 {(tts_w == 0.5).mean():.1%} "
      f"| at cap 3 {(tts_w == 3.0).mean():.1%} | quantiles 5/25/50/75/95% {np.round(q, 3).tolist()}")
report["tts_weights"] = {"clips": n_tts, "error_free": round(float((tts_w == 1.0).mean()), 4),
                         "at_floor": round(float((tts_w == 0.5).mean()), 4),
                         "at_cap": round(float((tts_w == 3.0).mean()), 4), "quantiles": np.round(q, 3).tolist()}


In [ ]:
# ---- 3. Agreement filter: stage-1 bulk decode of Iqra_train (train) and Iqra_TTS ----------------
model, info = parity.load_with_report(CKPT, DEVICE)
assert not any(info.values()), info
fe = AutoFeatureExtractor.from_pretrained("facebook/w2v-bert-2.0")   # as NB2 / NB14
train_split = load_from_disk(cfg.paths.train_arrow)["train"]
train_lab = [str(x) for x in list(train_split["phoneme_ref"])]

def bulk_per(ds, labels, n, name, chunk=512):
    # PER of the stage-1 greedy decode vs each label. Reads `chunk` clips, sorts them by
    # length (less padding), decodes them in padded batches of ~BATCH_SECONDS of audio.
    pers = np.full(n, np.nan)
    t0, done, next_report = time.time(), 0, 2000
    sub = ds.select(range(n)) if n < len(ds) else ds
    pending = []
    def run_chunk(items):
        items.sort(key=lambda x: len(x[1]))
        batch, secs = [], 0.0
        for k, (i, a) in enumerate(items):
            batch.append((i, a)); secs += len(a) / 16_000
            if secs >= BATCH_SECONDS or k == len(items) - 1:
                outs = posteriors.batch_logits(model, fe, [x[1] for x in batch], DEVICE, fp16=True)
                for (j, _), lg in zip(batch, outs):
                    pers[j] = weights.clip_per(labels[j].split(), greedy_decode(lg, ID2TOK, BLANK)[0])
                batch, secs = [], 0.0
    for i, arr, sr in data.iter_audio(sub):
        pending.append((i, data.to_16k(arr, sr)))
        if len(pending) == chunk:
            run_chunk(pending); done += len(pending); pending = []
            if done >= next_report:
                rate = done / (time.time() - t0)
                print(f"  {name}: {done}/{n} clips | {rate:.1f} clips/s | projected total {n / rate / 60:.0f} min")
                next_report = done + 20000
    if pending:
        run_chunk(pending); done += len(pending)
    assert not np.isnan(pers).any(), f"{name}: {int(np.isnan(pers).sum())} clips not decoded"
    print(f"  {name}: {n} clips in {(time.time() - t0) / 60:.1f} min")
    return pers

n_train = 500 if SMOKE else len(train_split)
per_train = bulk_per(train_split, train_lab, n_train, "iqra_train")
pd.DataFrame({"row": range(n_train), "per": per_train}).to_parquet(f"{OUT}/per_iqra_train.parquet", index=False)
per_tts = bulk_per(tts, tts_mis, n_tts, "iqra_tts")
pd.DataFrame({"row": range(n_tts), "per": per_tts}).to_parquet(f"{OUT}/per_iqra_tts.parquet", index=False)


In [ ]:
# ---- 4. Combine, check, write -------------------------------------------------------------------
agree_train = np.array([weights.agreement_weight(p) for p in per_train])
agree_tts = np.array([weights.agreement_weight(p) for p in per_tts])
sw = pd.concat([
    pd.DataFrame({"dataset": "iqra_train", "row": range(n_train), "per": per_train,
                  "agreement_weight": agree_train, "confusion_weight": 1.0}),
    pd.DataFrame({"dataset": "iqra_tts", "row": range(n_tts), "per": per_tts,
                  "agreement_weight": agree_tts, "confusion_weight": tts_w}),
], ignore_index=True)
sw["weight"] = sw["agreement_weight"] * sw["confusion_weight"]
sw["reason"] = np.where(sw["agreement_weight"] < 1, "agreement",
                        np.where(sw["confusion_weight"] != 1, "confusion", "none"))
sw.to_parquet(f"{OUT}/sample_weights.parquet", index=False)
down = {d: round(float((g["agreement_weight"] < 1).mean()), 4) for d, g in sw.groupby("dataset")}
down_all = round(float((sw["agreement_weight"] < 1).mean()), 4)
per_q = {d: np.round(np.nanquantile(g["per"], [0.5, 0.9, 0.99]), 3).tolist() for d, g in sw.groupby("dataset")}
print(f"down-weighted by agreement (PER > 0.30): {down} | all {down_all:.1%}  (done-when: under ~10%)")
print(f"PER quantiles 50/90/99%: {per_q}")
print(f"final weight: mean {sw['weight'].mean():.3f} | by dataset {sw.groupby('dataset')['weight'].mean().round(3).to_dict()}")
report["agreement"] = {"down_weighted": down, "down_weighted_all": down_all, "per_quantiles": per_q}
report["done_when"] = {"under_10pct_down_weighted": down_all < 0.10, "top10_printed": True}
with open(f"{OUT}/nb12_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=1, default=str)


In [ ]:
# ---- PASTE BACK ---------------------------------------------------------------------------------
print(json.dumps(report, ensure_ascii=False, indent=1, default=str))
print("files:", sorted(os.listdir(OUT)))
